## Introduction
   
<p style="text-align:justify; padding:20px;">
    In this notebook, we will focus on NLP. More spcifically on text generation. With the apparation of transformers in 2017, text generation have become one of the hot topics of datasience. To follow this trend, we are going to create new movie title from the Netflix dataset. To acheive this goal, we will deploy the distil version of the GPT2 transformer and fine tune the model. The notebook will be devided into 6 sections: 
</p>

* [Imports](#section-1)
* [Data preparation](#section-2)
* [Training the model](#section-3)
* [Generating titles](#section-4)

# <span>Title generation using GPT2 ✒️ </span>
<hr style="border-bottom: solid;background-color:light;color:black;">

<a id="section-1"></a>
# <span>1. Imports</span>
<hr style="border-bottom: solid;background-color:light;color:black;">

In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, random_split
from transformers import GPT2Tokenizer, TrainingArguments, Trainer, GPT2LMHeadModel

from sklearn.model_selection import train_test_split

import warnings
warnings.simplefilter("ignore")


In [2]:
tokenizer = GPT2Tokenizer.from_pretrained('distilgpt2', bos_token='<|startoftext|>',
                                          eos_token='<|endoftext|>', pad_token='<|pad|>')
model = GPT2LMHeadModel.from_pretrained('distilgpt2')
model.resize_token_embeddings(len(tokenizer))

Downloading:   0%|          | 0.00/0.99M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/762 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Downloading:   0%|          | 0.00/336M [00:00<?, ?B/s]

Embedding(50259, 768)

<a id="section-2"></a>
# <span>2. Data preparation</span>
<hr style="border-bottom: solid;background-color:light;color:black;">

In [3]:
titles = pd.read_csv('../input/netflix-shows/netflix_titles.csv')['title']

In [4]:
titles, test_titles = train_test_split(titles, test_size=10)

In [5]:
max_length = max([len(tokenizer.encode(title)) for title in titles])

In [6]:
class NetflixDataset(Dataset):
    def __init__(self, txt_list, tokenizer, max_length):
        self.input_ids = []
        self.attn_masks = []
        self.labels = []
        for txt in txt_list:
            encodings_dict = tokenizer('<|startoftext|>' + txt + '<|endoftext|>',
                                       max_length=max_length, padding="max_length")
            self.input_ids.append(torch.tensor(encodings_dict['input_ids']))
            self.attn_masks.append(torch.tensor(encodings_dict['attention_mask']))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.attn_masks[idx]

In [7]:
dataset = NetflixDataset(titles, tokenizer, max_length=max_length)
train_size = int(0.9 * len(dataset))
train_dataset, val_dataset = random_split(dataset, [train_size, len(dataset) - train_size])

<a id="section-3"></a>
# <span>3. Training the model</span>
<hr style="border-bottom: solid;background-color:light;color:black;">

In [8]:
training_args = TrainingArguments(output_dir='./results', num_train_epochs=1, logging_steps=100, save_steps=500,
                                  per_device_train_batch_size=1, per_device_eval_batch_size=1,
                                  warmup_steps=10, weight_decay=0.05, logging_dir='./logs', report_to = 'none')

In [ ]:
Trainer(model=model,  args=training_args, train_dataset=train_dataset,
        eval_dataset=val_dataset, data_collator=lambda data: {'input_ids': torch.stack([f[0] for f in data]),
                                                              'attention_mask': torch.stack([f[1] for f in data]),
                                                              'labels': torch.stack([f[0] for f in data])}).train()

***** Running training *****
  Num examples = 7917
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 1
  Gradient Accumulation steps = 1
  Total optimization steps = 7917


Step,Training Loss
100,2.179200
200,0.873300
300,0.750200
400,0.750500
500,0.796000
600,0.781100
700,0.697700
800,0.756500
900,0.752000
1000,0.734000


Saving model checkpoint to ./results/checkpoint-500
Configuration saved in ./results/checkpoint-500/config.json
Model weights saved in ./results/checkpoint-500/pytorch_model.bin
Saving model checkpoint to ./results/checkpoint-1000
Configuration saved in ./results/checkpoint-1000/config.json
Model weights saved in ./results/checkpoint-1000/pytorch_model.bin
Saving model checkpoint to ./results/checkpoint-1500
Configuration saved in ./results/checkpoint-1500/config.json
Model weights saved in ./results/checkpoint-1500/pytorch_model.bin
Saving model checkpoint to ./results/checkpoint-2000
Configuration saved in ./results/checkpoint-2000/config.json
Model weights saved in ./results/checkpoint-2000/pytorch_model.bin
Saving model checkpoint to ./results/checkpoint-2500
Configuration saved in ./results/checkpoint-2500/config.json
Model weights saved in ./results/checkpoint-2500/pytorch_model.bin
Saving model checkpoint to ./results/checkpoint-3000
Configuration saved in ./results/checkpoint-3

TrainOutput(global_step=7917, training_loss=0.7505495407010052, metrics={'train_runtime': 315.1014, 'train_samples_per_second': 25.125, 'train_steps_per_second': 25.125, 'total_flos': 82828773384192.0, 'train_loss': 0.7505495407010052, 'epoch': 1.0})

<a id="section-4"></a>
# <span>4. Generating titles</span>
<hr style="border-bottom: solid;background-color:light;color:black;">

In [ ]:
results = []
for title in test_titles:
    new_titles = {
        'seed': title.split()[0],
        'predictions': []
    }
    generated = tokenizer("<|startoftext|> "+ title.split()[0], return_tensors="pt").input_ids.cuda()
    sample_outputs = model.generate(generated,no_repeat_ngram_size = 1,num_beams=20, num_return_sequences=2)

    new_titles['predictions'] = sample_outputs
    results.append(new_titles)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generati

In [11]:
for new_title in results:
    print(f"seed: {new_title['seed']}")
    for i, pred in enumerate(new_title['predictions']):
        print(f"{i+1}: {tokenizer.decode(pred, skip_special_tokens=True)}")

seed: Holy
1:  Holy Grail
2:  Holy Eyes
seed: Chupan
1:  Chupan
2:  Chupanika
seed: BASEketball
1:  BASEketball
2:  BASEketball 2
seed: Bo
1:  Booby-Doo
2:  BoJack Horseman
seed: Doom:
1:  Doom: The Movie
2:  Doom: The End of the World
seed: Roohi
1:  Roohi
2:  Roohi Hai
seed: The
1:  The Dark Side of the Moon
2:  The Dark Side of the Moon (Telugu Version)
seed: HALO
1:  HALO
2:  HALO!
seed: First
1:  First Lady
2:  First Love
seed: Tremors
1:  Tremors
2:  Tremors 2


<a id="section-5"></a>
# <span>Work in progress</span>
<hr style="border-bottom: solid;background-color:light;color:black;">

<p style="text-align:justify; padding:20px;">
    If you have any suggestions on how I could improve this notebook, please let me know :D
</p>
